# Exact-integer update-cost smoke v1

One Adam update, then save, reload, and resource logging. The 64-task pack is not training data and is not scored. The mlx and transformers backends are not replicates. This notebook does not run the five-arm comparison as training; that entry point exits 2.

Use a GPU runtime. The commands match `docs/heldout-exact-integer-update-cost-smoke-v1.md`.

In [ ]:
import os

os.environ.setdefault("VARE_REPO", "https://github.com/mitukx/VARE.git")
os.environ.setdefault("VARE_REV", "cursor/exact-integer-update-cost-smoke-af75")
print(os.environ["VARE_REPO"], os.environ["VARE_REV"])


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/vare-update-cost-smoke-v1")
if not (repo / ".git").exists():
    subprocess.check_call(["git", "clone", os.environ["VARE_REPO"], str(repo)])
subprocess.check_call(["git", "fetch", "origin", os.environ["VARE_REV"]], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "/content/vare-update-cost-smoke-v1",
        "transformers",
        "accelerate",
        "peft",
        "huggingface_hub",
    ]
)


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/vare")
root.mkdir(parents=True, exist_ok=True)
output = root / "heldout-exact-integer-update-cost-smoke-v1-transformers"
output.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(root / "huggingface-cache")
os.environ["HF_HUB_CACHE"] = str(root / "huggingface-cache" / "hub")
print("output", output)


In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-update-cost-smoke-v1")
expected = {
    "protocols/heldout_exact_integer_update_cost_smoke_v1.json": "0d6959238b63b3b9a08b8db49c52a0905bbc51f617cd4b1ba8d5241da3c940d0",
    "benchmarks/heldout/exact_integer_v1/tasks.jsonl": "189b5e482b0f8c2b40ff36e0e1a84827699f348c11981a0ff912ce67323fd7fc",
    "results/heldout-exact-integer-base-gate-v2/m1-mlx-89f671c/manifest.json": "ee582b61fc2ca21d45bacb829c728bf47118d987f302b6f0f2eb8aeb82571278",
}
for name, digest in expected.items():
    got = hashlib.sha256((repo / name).read_bytes()).hexdigest()
    print(name, got)
    assert got == digest
completed = subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/run_exact_integer_update_cost_smoke_v1.py"),
        "--backend",
        "transformers",
        "--output",
        str(output),
    ],
    cwd=repo,
)
print("smoke_exit", completed.returncode)
refused = subprocess.run(
    [sys.executable, str(repo / "scripts/format_vs_correctness_comparison_v1.py")],
    cwd=repo,
)
print("comparison_exit", refused.returncode)
